# AI Product Recommendation Engine -- IoT Edition
Moteur intelligent & adaptatif : recommande des produits **similaires (meme type)** et **complementaires**, avec des regles editables via un fichier CSV.

In [ ]:
!pip install pandas numpy scikit-learn sentence-transformers -q
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.neighbors import NearestNeighbors
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
import re, unicodedata, warnings
warnings.filterwarnings("ignore")
try:
    from sentence_transformers import SentenceTransformer
    HAS_ST = True
    print("OK Dependances -- mode semantique (sentence-transformers)")
except Exception:
    HAS_ST = False
    print("Repli automatique sur TF-IDF (sentence-transformers indisponible)")

## Etape 1 : Chargement des donnees

In [ ]:
candidate_files = [
    Path("inventory_export.csv"),
    Path("data/inventory_export.csv"),
    Path("/content/inventory_export.csv"),
    Path("/content/drive/MyDrive/inventory_export.csv"),
]
csv_path = next((p for p in candidate_files if p.exists()), None)
if csv_path is None:
    try:
        from google.colab import files
        print("Selectionne inventory_export.csv a uploader...")
        uploaded = files.upload()
        csv_path = Path(list(uploaded.keys())[0])
    except ModuleNotFoundError:
        raise FileNotFoundError("inventory_export.csv introuvable.")
df_raw = pd.read_csv(csv_path)
print(f"OK Charge depuis : {csv_path}")
print(f"OK {df_raw.shape[0]} lignes, {df_raw['Handle'].nunique()} produits uniques")
df_raw.head(3)

## Etape 2 : Nettoyage + Categorisation complete (objectif 0% 'autre')

In [ ]:
df = df_raw.copy()
df.columns = df.columns.str.lower().str.strip()

def normalize_text(text):
    """Texte filtre (sans mots-outils, sans lettres seules) -- pour categories + TF-IDF."""
    text = "" if pd.isna(text) else str(text)
    text = unicodedata.normalize("NFKD", text).encode("ascii","ignore").decode("ascii").lower()
    text = re.sub(r"[^a-z0-9\s]"," ",text)
    stop = {"de","des","du","la","le","les","un","une","et","au","aux","pour","avec","sur","dans","par","ou","en"}
    return " ".join(t for t in text.split() if len(t)>1 and t not in stop)

def norm_raw(text):
    """Texte brut (GARDE les lettres seules : 'type c', 'pile d') -- pour l'extraction d'attributs."""
    text = "" if pd.isna(text) else str(text)
    text = unicodedata.normalize("NFKD", text).encode("ascii","ignore").decode("ascii").lower()
    return re.sub(r"[^a-z0-9]+"," ",text).strip()

# ============================================================================
# BRIQUE 0 -- CATEGORISATION COMPLETE (objectif : 0 % de "autre")
# ============================================================================
CATEGORY_RULES = {
 "capteur":     ["capteur","capteurs","senseur","sensor","dht11","dht22","ds18b20","hc-sr04","hcsr04","pir","ultrason","mpu6050","mpu9250","bmp","bme280","mq2","mq3","mq135","ldr","acs712","sct013","gy-","lm35","tof","photoresistance","accelerometre","gyroscope","camera","debimetre","yf-s201","flow","ad8232","ecg","hc-sr501","encodeur","mlx","reed","hx711","load cell","fin de course"],
 "carte":       ["raspberry","rpi","arduino","esp32","esp8266","nodemcu","wemos","microbit","stm32","atmega","jetson","wroom","attiny","carte developpement","mega2560","leonardo"],
 "mobilite":    ["voiture","telecommandee","telecommande","drone","quadcopter","helices","helice","roue","roues","chassis","4wd","2wd","robot","bateau","avion","mecanum","suiveur"],
 "moteur":      ["moteur","motor","servo","servomoteur","stepper","nema17","nema23","28byj","pompe","ventilateur","brushless","pas a pas","mg90","mg90s","mg995","mg996","sg90","ds3218","actionneur","vibreur"],
 "led":         ["led","ruban","bande","ampoule","lampe","lumiere","spectre","rgb","neon","matrice","ws2812","cob","horticole","projecteur","spot","luminaire","strip"],
 "batterie":    ["pile","piles","batterie","battery","18650","21700","26650","14500","lipo","li-po","lithium","accu","alcaline","alkaline","nimh","ni-mh","vape","cr2032","cr2016","cr1620","r20","r14"],
 "chargeur":    ["chargeur","charging","chargement","tp4056","imax"],
 "alimentation":["alimentation","tension","regulateur","regulator","convertisseur","transformateur","decoupage","buck","boost","abaisseur","elevateur","pwm","7805","79l12","onduleur","step-down","step-up","fusible"],
 "rf":          ["433mhz","rf","emetteur","recepteur","bluetooth","ble","wifi","gsm","gprs","gps","nrf24","lora","zigbee","sim800","sim900","antenne","hm-10","nfc","rfid"],
 "electronique":["module","circuit","relais","relay","lcd","oled","hdmi","i2c","spi","74hc","mosfet","registre","potentiometre","potentiometer","ecran","interrupteur","bouton","contacteur","rtc","driver","l298","shield","bouclier","pcb","prototype","perforee","controleur","cnc","gravure","expansion","stc-1000","uln2003","afficheur","segments","tm1637","max7219","clavier","membrane","ft232","rs232","cp2102","ch340","keypad","joystick","dip switch","horloge","ne555"],
 "composant":   ["diode","diodes","resistance","resistances","condensateur","transistor","zener","quartz","cristal","inductance","1n4148","1n4007","thyristor","triac","optocoupleur","triode","optotriac"],
 "connectique": ["cable","cables","fil","fils","connecteur","connecteurs","cosses","cosse","jumper","cavalier","dupont","barette","barrette","prise","borne","bornier","header","nappe","gaine","thermoretractable","domino","carte memoire","micro sd","microsd","sd card","cordon","rallonge","serre cable"],
 "mesure":      ["multimetre","testeur","oscilloscope","amperemetrique","pince ampere","compteur","balance","luxmetre","thermometre","wattmetre","pzem","ph metre","electrode","sonde","frequencemetre","generateur de signal"],
 "soudure":     ["souder","soudure","etain","flux","panne","dessoudage","fer a souder","desoudage"],
 "solaire":     ["solaire","photovoltaique","photovolta","mppt","panneau solaire"],
 "audio":       ["haut-parleur","haut parleur","microphone","amplificateur","buzzer","speaker","ampli","sonore","hp","mp3","jack audio"],
 "outillage":   ["tournevis","cle","cles","clef","pince","pinces","scie","cutter","lame","lames","foret","forets","marteau","coffret","outil","outils","douille","douilles","embout","embouts","percage","brucelle","mandrin","cliquet","perceuse","visseuse","meuleuse","burin","lime","etau","cle mixte","jeu de","pistolet","colle","peinture","disque","brosse","meule","nettoyeur","levage","electrogene","laser","niveau","compresseur","ponceuse","rabot","agrafeuse","riveteuse","decapeur","imprimante","creality","filament","truelle","spatule","pinceau","rouleau","echelle","escabeau","cric","palan","aspirateur","disqueuse","sangle"],
 "mecanique":   ["vis","ecrou","ecrous","boulon","rondelle","rondelles","rivet","roulement","engrenage","courroie","poulie","ressort","rail","profile","entretoise","boitier","coque","dissipateur","heatsink","radiateur","colonnette","equerre","charniere","glissiere","tige filetee"],
 "electrique":  ["disjoncteur","contacteur","sectionneur","chint","parafoudre","goulotte","armoire"],
}
ACCESSORY = {"lcd","oled","ecran","afficheur","ruban","boitier","coque","dissipateur",
             "ventilateur","camera","shield","transformateur","adaptateur","alimentation","chargeur"}
CAT_ORDER = ["chargeur","batterie","led","alimentation","moteur","rf","electronique","composant",
             "connectique","mesure","soudure","solaire","audio","outillage","mecanique","electrique"]

TOOL_BRANDS = {"total","wadfow","harden","yato","tolsen","ingco","stanley","bosch","makita",
               "dewalt","milwaukee","creality","deli"}
def is_tool_brand(title):
    return any(b in norm_raw(title).split() for b in TOOL_BRANDS)

def _has(n, words, kws):
    for k in kws:
        if " " in k:
            if k in n: return True
        elif len(k) <= 3:
            if k in words: return True
        else:
            if any(k in w for w in words) or k in n: return True
    return False

def infer_category(title):
    n = normalize_text(title); words = set(n.split()); wr = set(norm_raw(title).split())
    if _has(n, words, CATEGORY_RULES["mobilite"]): return "mobilite"
    if _has(n, words, CATEGORY_RULES["capteur"]):  return "capteur"
    if _has(n, words, CATEGORY_RULES["carte"]) and not (wr & ACCESSORY): return "carte"
    for cat in CAT_ORDER:
        if _has(n, words, CATEGORY_RULES[cat]): return cat
    if is_tool_brand(title): return "outillage"
    return "autre"

# ============================================================================
# SPECS
# ============================================================================
SPEC_PATTERNS = {
 "capacity_mah":(r'\b(\d+(?:[.,]\d+)?)\s*mah\b',50,100000),
 "capacity_ah": (r'\b(\d+(?:[.,]\d+)?)\s*ah\b',0.2,500),
 "voltage_v":   (r'\b(\d+(?:[.,]\d+)?)\s*v\b',0.5,600),
 "power_w":     (r'\b(\d+(?:[.,]\d+)?)\s*w\b',0.1,5000),
 "count":       (r'\b(\d+)\s*(?:pcs|pieces|broches|pin|leds?|canaux|ch)\b',1,10000),
}
def extract_specs(title):
    t = unicodedata.normalize("NFKD", str(title)).encode("ascii","ignore").decode("ascii").lower().replace("*"," ")
    out={}
    for name,(pat,lo,hi) in SPEC_PATTERNS.items():
        v=[float(x.replace(",",".")) for x in re.findall(pat,t)]; v=[x for x in v if lo<=x<=hi]
        if v: out[name]=max(v)
    return out
UPGRADE_SPECS=["capacity_mah","capacity_ah","power_w","count"]
def primary_spec(specs):
    for k in UPGRADE_SPECS:
        if k in specs: return k,specs[k]
    return None,0.0

# ============================================================================
# BRIQUE 1 -- ATTRIBUTS
# ============================================================================
LIION_FF=["18650","21700","26650","14500","16340","18500","20700"]; COIN_FF=["cr2032","cr2016","cr1620","cr2025","cr2450","lr44","ag13"]
def a_form_factor(n):
    for f in LIION_FF:
        if f in n: return f
    for f in COIN_FF:
        if f in n: return "coin"
    if re.search(r'\b(rl20|pile d)\b',n): return "d"
    if re.search(r'\b(rl14|pile c)\b',n): return "c"
    if re.search(r'\b(aaa|lr03|r03)\b',n): return "aaa"
    if re.search(r'\b(aa|lr6|r6)\b',n): return "aa"
    if re.search(r'\b9v\b',n): return "9v"
    if re.search(r'\bnema\s?17\b',n): return "nema17"
    if re.search(r'\bnema\s?23\b',n): return "nema23"
    if "5mm" in n: return "5mm"
    if "3mm" in n: return "3mm"
    for c in ["5050","2835","3528"]:
        if c in n: return c
    return None
def a_chemistry(n,ff):
    if "lipo" in n or "li-po" in n or "polymer" in n: return "lipo"
    if ff in LIION_FF or "li-ion" in n or "liion" in n or "lithium" in n: return "lithium"
    if "nimh" in n or "ni-mh" in n: return "nimh"
    if "alcaline" in n or "alkaline" in n or ff in ("d","c","aa","aaa","9v"): return "alkaline"
    if "plomb" in n or "agm" in n or "ultracell" in n or "acide" in n: return "lead"
    if "vape" in n or "pod" in n: return "vape"
    if ff=="coin": return "lithium"
    return None
def a_connector(n):
    if "type-c" in n or "type c" in n or "usb-c" in n or "usb c" in n: return "usb_c"
    if "micro usb" in n or "micro-usb" in n or "microusb" in n: return "usb_micro"
    if "type-b" in n or "type b" in n or "usb-b" in n: return "usb_b"
    if "xt60" in n: return "xt60"
    if "jst" in n: return "jst"
    if "hdmi" in n: return "hdmi"
    if "rj45" in n or "ethernet" in n: return "rj45"
    if "jack" in n: return "jack"
    if "usb" in n: return "usb_a"
    return None
def a_connectivity(n):
    if "wifi" in n or "wi-fi" in n: return "sans" if ("sans wifi" in n or "without wifi" in n) else "wifi"
    if "bluetooth" in n or re.search(r'\bble\b',n): return "bluetooth"
    if "433" in n or "nrf24" in n or "lora" in n or "zigbee" in n or re.search(r'\brf\b',n): return "rf"
    if "gsm" in n or "gprs" in n: return "gsm"
    if "gps" in n: return "gps"
    if "sans fil" in n: return "wireless"
    return None
def a_control(n):
    if "application" in n or re.search(r'\bappli\b',n) or re.search(r'\bapp\b',n): return "app"
    if "wifi" in n: return "wifi"
    if "bluetooth" in n: return "bluetooth"
    if "telecommand" in n or "radiocommand" in n or re.search(r'\brc\b',n): return "rc"
    return None
def a_vehicle(n):
    w=set(n.split())
    if "voiture" in n or "automobile" in n: return "voiture"
    if "drone" in n or "quadcopter" in n: return "drone"
    if "bateau" in n: return "bateau"
    if "avion" in n: return "avion"
    if "moto" in w: return "moto"
    if "tank" in w: return "char"
    if "robot" in n: return "robot"
    return None
def a_board(n):
    w=set(n.split())
    if "esp32" in n: return "esp32"
    if "esp8266" in n or "nodemcu" in n or "wemos" in n or "esp-12" in n: return "esp8266"
    if "raspberry" in n or "rpi" in w or "pi" in w: return "raspberry"
    if "arduino" in n: return "arduino"
    if w & {"uno","nano","leonardo","mega"}: return "arduino"
    if "microbit" in n: return "microbit"
    if "stm32" in n: return "stm32"
    if "jetson" in n: return "jetson"
    return None
def a_led_form(n):
    if "ruban" in n or "bande" in n or "strip" in n: return "strip"
    if "ampoule" in n or "spot" in n or re.search(r'\be14\b',n) or re.search(r'\be27\b',n) or "gu10" in n: return "bulb"
    if "matrice" in n or "ws2812" in n or "8x8" in n or "neopixel" in n: return "matrix"
    if "cob" in n or "horticole" in n or "spectre" in n: return "cob"
    if "infrarouge" in n or "850nm" in n or "940nm" in n or re.search(r'\bir\b',n): return "ir"
    if "afficheur" in n or "7 segment" in n or "segments" in n: return "display"
    if "neon" in n: return "neon"
    if "5mm" in n or "3mm" in n: return "discrete"
    if "lampe" in n or "projecteur" in n or "luminaire" in n: return "bulb"
    return None
def a_voltage_bucket(specs):
    v=specs.get("voltage_v")
    if v is None: return None
    for b in (3.3,5,12,24,220):
        if abs(v-b)<=max(0.6,b*0.12): return b
    return round(v)
def a_psu(n):
    if "buck" in n or "abaisseur" in n or "step down" in n: return "buck"
    if "boost" in n or "elevateur" in n or "step up" in n: return "boost"
    if "decoupage" in n or "transformateur" in n or "ac dc" in n: return "acdc"
    if "pwm" in n: return "pwm"
    return None
def a_rftech(n):
    if "433" in n: return "433"
    if "bluetooth" in n or re.search(r'\bble\b',n) or "hm-10" in n: return "bluetooth"
    if "wifi" in n: return "wifi"
    if "gsm" in n or "gprs" in n or "sim800" in n: return "gsm"
    if "gps" in n: return "gps"
    if "lora" in n: return "lora"
    if "zigbee" in n: return "zigbee"
    if "nrf24" in n: return "nrf24"
    if "nfc" in n or "rfid" in n: return "nfc"
    return None
def a_component(n):
    for c in ["resistance","condensateur","transistor","zener","diode","quartz","inductance","optocoupleur","thyristor","triac"]:
        if c in n: return c
    return None
def a_module_fn(n):
    # reconnait la FONCTION d'un circuit / module (registre, timer, rtc, driver, ampli...)
    if "relais" in n or "relay" in n: return "relais"
    if "registre" in n or "decalage" in n or "shift register" in n or "74hc595" in n or "74hc164" in n or "74hc165" in n: return "registre"
    if "rtc" in n or "horloge" in n or "ds1302" in n or "ds3231" in n: return "rtc"
    if "ne555" in n or "timer" in n or re.search(r'\b555\b',n): return "timer"
    if "lcd" in n or "oled" in n or "afficheur" in n or "ecran" in n: return "display"
    if "l298" in n or "uln2003" in n or "driver" in n or "darlington" in n: return "driver"
    if "amplificateur" in n or "opamp" in n or "lm358" in n or "lm741" in n or "tda" in n: return "ampli"
    if "potentiometre" in n or "potentiometer" in n: return "potentiometre"
    if "optocoupleur" in n: return "opto"
    if "convertisseur" in n: return "convertisseur"
    if "interrupteur" in n: return "interrupteur"
    if "bouton" in n: return "bouton"
    if re.search(r'\b74(hc|hct|ls)\d',n): return "logique"
    return None
def a_motor(n):
    if "servo" in n or re.search(r'\b(mg90|mg90s|mg995|mg996|sg90|ds3218)\b',n): return "servo"
    if "stepper" in n or "nema" in n or "28byj" in n or "pas a pas" in n: return "stepper"
    if "pompe" in n: return "pompe"
    if "ventilateur" in n: return "ventilateur"
    if "moteur" in n or "motor" in n: return "dc"
    return None
def a_sensor(n):
    if "camera" in n: return "camera"
    if "dht" in n or "ds18b20" in n or "lm35" in n or "temperature" in n or "thermom" in n: return "temperature"
    if "hc-sr04" in n or "ultrason" in n or "sharp" in n or "tof" in n or "distance" in n: return "distance"
    if "pir" in n or "mouvement" in n or "motion" in n or "hc-sr501" in n: return "motion"
    if "mq" in n or "gaz" in n or "co2" in n or "pollution" in n: return "gaz"
    if "bmp" in n or "bme" in n or "pression" in n: return "pression"
    if "acs712" in n or "sct" in n or "courant" in n: return "courant"
    if "mpu" in n or "gyro" in n or "accel" in n: return "imu"
    if "ldr" in n or "lux" in n: return "lumiere"
    if "humidite" in n or "pluie" in n: return "humidite"
    return None

def extract_attrs(title, specs):
    n = norm_raw(title); ff = a_form_factor(n)
    return {"form_factor":ff,"chemistry":a_chemistry(n,ff),"connector":a_connector(n),
            "connectivity":a_connectivity(n),"control":a_control(n),"vehicle":a_vehicle(n),
            "board":a_board(n),"led_form":a_led_form(n),"voltage_bucket":a_voltage_bucket(specs),
            "psu":a_psu(n),"rftech":a_rftech(n),"component":a_component(n),
            "module_fn":a_module_fn(n),"motor":a_motor(n),"sensor":a_sensor(n)}

# ============================================================================
# Construction des colonnes
# ============================================================================
df["product_handle"] = df["handle"].astype(str).str.strip()
df["product_title"]  = df["title"].astype(str).str.strip()
df["sku"]            = df["sku"].astype(str).str.strip() if "sku" in df.columns else ""

for col in ["available (not editable)","on hand (current)","on hand (new)"]:
    if col in df.columns:
        df[col] = df[col].replace("not stocked",0)
        df[col] = pd.to_numeric(df[col],errors="coerce").fillna(0).astype(int)

df["clean_title"] = df["product_title"].apply(normalize_text)
df["category"]    = df["product_title"].apply(infer_category)

# Repli INTELLIGENT : tout produit encore "autre" herite de la categorie de son plus proche voisin
_mask = df["category"] == "autre"
if _mask.any():
    _known = df.loc[~_mask]
    _v  = TfidfVectorizer(max_features=3000).fit(df["clean_title"])
    _nn = NearestNeighbors(n_neighbors=1, metric="cosine").fit(_v.transform(_known["clean_title"]))
    _, _ind = _nn.kneighbors(_v.transform(df.loc[_mask, "clean_title"]))
    df.loc[_mask, "category"] = _known["category"].to_numpy()[_ind[:, 0]]

df["specs"]   = df["product_title"].apply(extract_specs)
df["attrs"]   = df.apply(lambda r: extract_attrs(r["product_title"], r["specs"]), axis=1)
df["is_tool"] = df["product_title"].apply(is_tool_brand)

uniq = df.drop_duplicates("product_handle")
print("OK Categorisation complete + attributs extraits")
dist = uniq["category"].value_counts()
print("\nDistribution des categories :")
print(dist.to_string())
print(f"\n   'autre' = {100*dist.get('autre',0)/len(uniq):.1f}%   "
      f"| produits avec >=1 attribut = {100*(uniq['attrs'].apply(lambda a: any(v is not None for v in a.values()))).mean():.0f}%")

In [ ]:
# === Affinage des categories : eviter les confusions du catalogue ===
_t = df["product_title"]

# 1) Accessoires AUDIO-AUTO (transmetteur FM, recepteur/adaptateur audio) -> 'audio'
_audio_acc = _t.str.contains(r"transmetteur fm|transmetteur audio|adaptateur audio|recepteur audio|mains.?libres",
                             case=False, na=False)
df.loc[_audio_acc, "category"] = "audio"

# 2) OUTILS-MOTEUR automobiles (le mot "moteur" = aussi "engine") -> 'outillage', PAS moteur electrique
_engine_tool = _t.str.contains(
    r"compressiometre|compression moteur|soupape|depose|calage|distribution|vilebrequin|culasse|injecteur|"
    r"support moteur|cale.?moteur|extracteur|arrache|demonte|cl[ée] moteur|testeur.*moteur",
    case=False, na=False)
df.loc[_engine_tool & (df["category"]=="moteur"), "category"] = "outillage"

# 3) Accessoires de BATTERIE (ne sont PAS des piles de remplacement) -> vraie categorie
df.loc[_t.str.contains(r"\btesteur\b", case=False, na=False) & (df["category"]=="batterie"), "category"] = "mesure"
df.loc[_t.str.contains(r"\bbms\b|carte de protection|protection.*batterie", case=False, na=False) & (df["category"]=="batterie"), "category"] = "composant"
df.loc[_t.str.contains(r"boitier|power bank|porte.?pile", case=False, na=False) & (df["category"]=="batterie"), "category"] = "mecanique"

_b = int((df.drop_duplicates('product_handle')['category']=='batterie').sum())
_m = int((df.drop_duplicates('product_handle')['category']=='moteur').sum())
print(f"OK Affinage : audio-auto, outils-moteur et accessoires batterie reclasses")
print(f"   batterie = {_b} piles reelles | moteur = {_m} moteurs electriques reels")

## Etape 3 : Regles de recommandation (fichier CSV editable)
Le fichier `regles_recommandation.csv` pilote la logique : criteres du 'meme type' + complements. Edite-le sans toucher au code.

In [ ]:
# ============================================================================
# REGLES DE RECOMMANDATION PILOTEES PAR UN FICHIER CSV (editable sans code)
#   regles_recommandation.csv dit, pour chaque categorie :
#     - attributs_similaire          : criteres qui definissent "meme type"  (SIMILAIRES)
#     - complement_categories        : ou chercher les complements           (COMPLEMENTAIRES)
#     - complement_categories_toutes : categories ou TOUS les produits sont de bons complements
#     - complement_mots_cles         : mots-cles d'un BON complement
#   -> edite le CSV (toi ou ton coach) puis re-execute, sans toucher au code.
# ============================================================================
RULES_CSV = Path("regles_recommandation.csv")

_DEFAULT_RULES = [
 {"categorie":"batterie","attributs_similaire":"form_factor;chemistry","complement_categories":"chargeur;composant;mecanique;connectique","complement_categories_toutes":"","complement_mots_cles":"bms;protection;support;porte pile;boitier;holder","exemple":"Batterie 18650 -> chargeur 18650 (form_factor) / BMS / support"},
 {"categorie":"chargeur","attributs_similaire":"chemistry;connector","complement_categories":"batterie;connectique","complement_categories_toutes":"","complement_mots_cles":"batterie;pile;accu","exemple":"Chargeur 18650 -> batteries 18650"},
 {"categorie":"led","attributs_similaire":"led_form;voltage_bucket","complement_categories":"alimentation;electronique;connectique","complement_categories_toutes":"","complement_mots_cles":"alimentation;transformateur;controleur;dimmer;connecteur ruban","exemple":"Ruban LED 12V -> alim 12V / controleur RGB"},
 {"categorie":"alimentation","attributs_similaire":"voltage_bucket;connector","complement_categories":"led;carte;electronique;connectique","complement_categories_toutes":"","complement_mots_cles":"ruban;led;module","exemple":"Alim 12V -> ruban LED 12V"},
 {"categorie":"carte","attributs_similaire":"board;connectivity","complement_categories":"capteur;connectique;alimentation;led;electronique;composant","complement_categories_toutes":"capteur","complement_mots_cles":"breadboard;plaque essai;dupont;jumper;cable usb;micro sd;carte memoire;ecran;oled;afficheur;resistance;alimentation","exemple":"Arduino -> breadboard / dupont / capteurs / alim 5V"},
 {"categorie":"capteur","attributs_similaire":"sensor","complement_categories":"carte;connectique;electronique","complement_categories_toutes":"carte","complement_mots_cles":"dupont;jumper;breadboard;arduino;esp32;raspberry;resistance","exemple":"Capteur DHT11 -> Arduino / fils dupont"},
 {"categorie":"moteur","attributs_similaire":"motor","complement_categories":"carte;electronique;alimentation;batterie","complement_categories_toutes":"","complement_mots_cles":"driver;l298;controleur;arduino;roue;helice;batterie","exemple":"Servo -> driver moteur / Arduino / batterie"},
 {"categorie":"mobilite","attributs_similaire":"vehicle;control","complement_categories":"batterie;chargeur;moteur;electronique;carte;connectique","complement_categories_toutes":"moteur","complement_mots_cles":"telecommande;radiocommande;roue;helice;chassis;lipo;li-ion;18650;nimh;driver;l298;controleur;esc;dupont","exemple":"Voiture RC -> moteur / driver / batterie LiPo / telecommande"},
 {"categorie":"rf","attributs_similaire":"rftech","complement_categories":"carte;alimentation;connectique","complement_categories_toutes":"","complement_mots_cles":"arduino;esp32;raspberry;antenne;dupont;module","exemple":"Module 433MHz -> Arduino / antenne"},
 {"categorie":"electronique","attributs_similaire":"module_fn","complement_categories":"carte;connectique;alimentation;composant","complement_categories_toutes":"","complement_mots_cles":"arduino;raspberry;dupont;breadboard;resistance;alimentation","exemple":"Relais -> Arduino / alimentation"},
 {"categorie":"composant","attributs_similaire":"component","complement_categories":"electronique;connectique;carte","complement_categories_toutes":"","complement_mots_cles":"breadboard;plaque essai;dupont;resistance;support;arduino","exemple":"Resistance -> breadboard / fils dupont"},
 {"categorie":"connectique","attributs_similaire":"connector","complement_categories":"carte;electronique","complement_categories_toutes":"","complement_mots_cles":"arduino;raspberry;module","exemple":"Cable dupont -> carte / module"},
 {"categorie":"mesure","attributs_similaire":"","complement_categories":"composant;connectique","complement_categories_toutes":"","complement_mots_cles":"sonde;electrode;pince;cable","exemple":"Multimetre -> sondes / pinces"},
 {"categorie":"soudure","attributs_similaire":"","complement_categories":"composant;connectique","complement_categories_toutes":"","complement_mots_cles":"etain;flux;fer;panne;support","exemple":"Fer a souder -> etain / flux"},
 {"categorie":"solaire","attributs_similaire":"voltage_bucket","complement_categories":"batterie;alimentation;chargeur","complement_categories_toutes":"","complement_mots_cles":"batterie;controleur;mppt;regulateur","exemple":"Panneau solaire -> batterie / MPPT"},
 {"categorie":"audio","attributs_similaire":"","complement_categories":"carte;alimentation;connectique","complement_categories_toutes":"","complement_mots_cles":"amplificateur;haut parleur;jack;arduino","exemple":"Micro -> ampli / Arduino"},
 {"categorie":"outillage","attributs_similaire":"","complement_categories":"","complement_categories_toutes":"","complement_mots_cles":"","exemple":"Outil -> aucun complement IoT"},
 {"categorie":"mecanique","attributs_similaire":"","complement_categories":"","complement_categories_toutes":"","complement_mots_cles":"","exemple":"Vis -> aucun complement IoT"},
 {"categorie":"electrique","attributs_similaire":"","complement_categories":"","complement_categories_toutes":"","complement_mots_cles":"","exemple":"Disjoncteur -> aucun complement IoT"},
]
if not RULES_CSV.exists():
    pd.DataFrame(_DEFAULT_RULES).to_csv(RULES_CSV, index=False)
    print(f"Fichier de regles cree : {RULES_CSV}  (edite-le pour regler les recommandations)")

_rules = pd.read_csv(RULES_CSV).fillna("")
def _spl(s): return [x.strip() for x in str(s).split(";") if x.strip()]

TYPE_ATTRIBUTES     = {r["categorie"]: _spl(r["attributs_similaire"])           for _,r in _rules.iterrows() if _spl(r["attributs_similaire"])}
COMPLEMENTARITY_MAP = {r["categorie"]: _spl(r["complement_categories"])         for _,r in _rules.iterrows() if _spl(r["complement_categories"])}
COMPLEMENT_BROAD    = {r["categorie"]: _spl(r["complement_categories_toutes"])  for _,r in _rules.iterrows()}
COMPLEMENT_KEYWORDS = {r["categorie"]: [normalize_text(k) for k in _spl(r["complement_mots_cles"])] for _,r in _rules.iterrows()}

print(f"OK Regles chargees depuis '{RULES_CSV.name}' -- {len(_rules)} categories")
print("   -> edite ce CSV puis re-execute a partir de l'Etape 5 (le moteur), sans toucher au code.")
_rules[["categorie","attributs_similaire","complement_categories","complement_mots_cles"]].head(6)

## Etape 4 : Agregation des produits

In [ ]:
products = df.drop_duplicates(subset=["product_handle"]).copy()
inv = df.groupby("product_handle").agg({"available (not editable)":"sum","on hand (current)":"sum","on hand (new)":"sum"}).reset_index()
inv.columns = ["product_handle","available_total","on_hand_current","on_hand_new"]
products = products.merge(inv, on="product_handle", how="left")
products["available_total"] = products["available_total"].fillna(0).astype(int)
products["product_profile"] = products["product_title"].fillna("") + " categorie " + products["category"].fillna("")
products = products[["product_handle","product_title","category","sku","specs","attrs","is_tool","clean_title","product_profile","available_total","on_hand_current","on_hand_new"]].reset_index(drop=True)
print(f"OK {len(products)} produits uniques | En stock: {len(products[products['available_total']>0])}")
products[["product_title","category","attrs","available_total"]].head(5)

### Token Hugging Face (optionnel, securise)
Modele public -> token non requis. Sinon : secret Colab ou fichier `.env` (jamais en dur, jamais pousse sur git).

In [ ]:
import os
def _load_dotenv(path=".env"):
    p = Path(path)
    if not p.exists():
        return
    for line in p.read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if line and not line.startswith("#") and "=" in line:
            k, v = line.split("=", 1)
            os.environ.setdefault(k.strip(), v.strip().strip(chr(34)).strip(chr(39)))
HF_TOKEN = None
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    _load_dotenv()
    HF_TOKEN = os.getenv("HF_TOKEN")
if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
    os.environ["HUGGINGFACE_HUB_TOKEN"] = HF_TOKEN
    print("Token Hugging Face charge (secret/.env).")
else:
    print("Aucun token HF -- pas grave, le modele est public.")

## Etape 5 : Moteur de recommandation

In [ ]:
class IoTRecommendationEngine:
    """
    Moteur de recommandation intelligent & adaptatif (regles pilotees par CSV).

      SIMILAIRES     : meme type (criteres adaptatifs par categorie), classes par
                       similarite semantique ; les meilleures specs en tete.
      COMPLEMENTAIRES: produits REELLEMENT compatibles (connecteur/tension/chimie/
                       mots-cles du CSV) ; jamais "ca se ressemble" seul, jamais un outil.

    Similarite EXACTE calculee sur tout le catalogue (pas seulement le top-K) -> classement
    plus juste. STRICT : rien plutot qu'un faux.
    """
    STRONG = re.compile(r'^(?=.*[a-z])(?=.*\d)[a-z0-9]{3,}$')

    def __init__(self, products_df, complementarity_map, complement_broad=None, complement_keywords=None):
        self.products = products_df.copy().reset_index(drop=True)
        self.comp_map = complementarity_map
        self.comp_broad = complement_broad or {}
        self.comp_keywords = complement_keywords or {}
        self.products["clean_profile"] = self.products["product_profile"].fillna("").apply(normalize_text)
        self.products["strong"] = self.products["clean_profile"].apply(
            lambda s: {w for w in s.split() if self.STRONG.match(w)})
        self.cat_to_idx = {c: list(g.index) for c, g in self.products.groupby("category")}
        self._cat   = self.products["category"].tolist()
        self._attrs = self.products["attrs"].tolist()
        self._tool  = self.products["is_tool"].tolist()
        self._stock = self.products["available_total"].tolist()
        self._strong= self.products["strong"].tolist()
        self._title_clean = self.products["clean_title"].tolist()   # titre seul (sans "categorie X")
        self._specs = self.products["specs"].tolist()

        if HAS_ST:
            print("Chargement du modele semantique...")
            self.model = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")
            self.X = self.model.encode([f"passage: {p}" for p in self.products["clean_profile"]],
                                       normalize_embeddings=True, show_progress_bar=False)
            self.is_sparse = False; mode = "semantique (sentence-transformers)"
        else:
            self.vec = TfidfVectorizer(max_features=5000, ngram_range=(1,2))   # vecteurs L2-normalises
            self.X = self.vec.fit_transform(self.products["clean_profile"])
            self.is_sparse = True; mode = "TF-IDF (repli hors-ligne)"
        print(f"OK Moteur pret -- {len(self.products)} produits indexes -- mode {mode}")

    def get_product_index(self, q):
        if isinstance(q, (int, np.integer)):
            return int(q) if 0 <= q < len(self.products) else None
        m = self.products[self.products["product_title"].str.contains(re.escape(str(q)), case=False, na=False)]
        return int(m.index[0]) if len(m) else None

    def _sims(self, i):
        """Similarite cosine EXACTE entre le produit i et TOUS les produits (vecteurs normalises)."""
        xi = self.X[i]
        if self.is_sparse:
            return (self.X @ xi.T).toarray().ravel()
        return self.X @ xi

    def _same_type(self, src_cat, src_attrs, src_tool, j):
        if bool(self._tool[j]) != bool(src_tool): return False
        req = [a for a in TYPE_ATTRIBUTES.get(src_cat, []) if src_attrs.get(a) is not None]
        if req:
            return all(self._attrs[j].get(a) == src_attrs[a] for a in req)
        return None

    def _rows(self, idx_list, S):
        cols = ["product_title","category","attrs","specs","available_total"]
        if not idx_list:
            return pd.DataFrame(columns=cols+["similarity"])
        out = self.products.loc[idx_list, cols].copy()
        out["similarity"] = [round(float(S[j]),3) for j in idx_list]
        return out.reset_index(drop=True)

    def recommend_smart(self, query, n=3, in_stock_only=True, comp_min=0.12):
        i = self.get_product_index(query)
        if i is None: return None
        cat = self._cat[i]; sa = self._attrs[i]; stool = self._tool[i]; src_strong = self._strong[i]
        pk, pv = primary_spec(self._specs[i])
        S = self._sims(i)   # similarite exacte vs tout le catalogue

        def ok_stock(j): return (not in_stock_only) or self._stock[j] > 0

        # ===== SIMILAIRES : meme type (strict adaptatif) =====
        same = []
        for j in self.cat_to_idx.get(cat, []):
            if j == i or not ok_stock(j): continue
            st = self._same_type(cat, sa, stool, j)
            if st is True:
                same.append(j)
            elif st is None:   # pas d'attribut discriminant -> repli : sim forte + token identite partage
                if bool(self._tool[j])==bool(stool) and S[j] >= 0.40 and (src_strong & self._strong[j]):
                    same.append(j)
        ups = []
        if pk:
            ups = [j for j in same if primary_spec(self._specs[j])[0]==pk and primary_spec(self._specs[j])[1] > pv]
            ups = sorted(ups, key=lambda j:(primary_spec(self._specs[j])[1], S[j]), reverse=True)
        alts = sorted([j for j in same if j not in ups], key=lambda j: S[j], reverse=True)
        similars = (ups + alts)[:n]

        # ===== COMPLEMENTAIRES : reellement compatibles (regles CSV) =====
        comp = []
        if not stool:
            comp_cats = self.comp_map.get(cat, [])
            broad = set(self.comp_broad.get(cat, []))
            ckw = self.comp_keywords.get(cat, [])
            volt_ok = cat not in ("batterie","chargeur")
            scored = []
            for ci, cc in enumerate(comp_cats):
                cc_broad = cc in broad
                for j in self.cat_to_idx.get(cc, []):
                    if j == i or not ok_stock(j) or self._tool[j]: continue
                    ja = self._attrs[j]; hard = 0.0
                    if sa.get("connector") and ja.get("connector")==sa["connector"]: hard += 0.5
                    if sa.get("form_factor") and sa["form_factor"] in self._title_clean[j]: hard += 0.6
                    if sa.get("chemistry") and ja.get("chemistry")==sa["chemistry"]: hard += 0.3
                    if volt_ok and sa.get("voltage_bucket") and ja.get("voltage_bucket")==sa["voltage_bucket"]: hard += 0.4
                    if ckw and any(k in self._title_clean[j] for k in ckw): hard += 0.5
                    if cc_broad: hard += 0.4
                    tok = 1 if (src_strong & self._strong[j]) else 0
                    if hard <= 0 and not tok: continue   # compatibilite REELLE requise (pas la sim seule)
                    # classement : compatibilite + identite + sens semantique (poids fort sur la sim)
                    score = 0.40*min(hard,1.0) + 0.20*tok + 0.40*float(S[j])
                    if score >= comp_min: scored.append((j, score, ci))
            comp = [j for j,_,_ in sorted(scored, key=lambda x:(-x[1], x[2]))[:n]]

        return {"source_idx":i, "source_title":self.products.at[i,"product_title"], "source_cat":cat,
                "source_attrs":{k:v for k,v in sa.items() if v is not None},
                "primary_spec":(pk,pv), "is_tool":bool(stool),
                "similars":self._rows(similars,S), "complements":self._rows(comp,S),
                # compat retro (anciens noms) :
                "upgrade":self._rows(ups[:n],S), "alternative":self._rows(alts[:n],S), "goes_with":self._rows(comp,S)}


engine = IoTRecommendationEngine(products, COMPLEMENTARITY_MAP, COMPLEMENT_BROAD, COMPLEMENT_KEYWORDS)

## Etape 6 : Tests

In [ ]:
def _attrs_str(a):
    a = {k:v for k,v in (a or {}).items() if v is not None}
    return ", ".join(f"{k}={v}" for k,v in a.items()) if a else "--"

def show_smart(query, n=3, in_stock_only=True):
    """Affiche 2 sections : PRODUITS SIMILAIRES + PRODUITS COMPLEMENTAIRES."""
    res = engine.recommend_smart(query, n=n, in_stock_only=in_stock_only)
    if res is None:
        print(f"X Produit introuvable : '{query}'"); return
    print("="*100)
    print(f"SELECTION : {res['source_title'][:80]}   [{res['source_cat']}]")
    print("="*100)
    for titre, dfb in [("🔁 PRODUITS SIMILAIRES",      res["similars"]),
                       ("🧩 PRODUITS COMPLEMENTAIRES", res["complements"])]:
        print(f"\n{titre} :")
        if len(dfb) == 0:
            print("   (aucun en stock)"); continue
        for _, r in dfb.iterrows():
            print(f"   - [{r['category']:<12}] {r['product_title'][:58]:<58}  sim={r['similarity']:.2f}")
    print()

# TEST 1 -- Batterie 18650
show_smart("18650 3.7V 2600MAH")

In [ ]:
# TEST 2 -- Cartes : Arduino (sans wifi) != ESP32 (wifi)
show_smart("Arduino UNO")
show_smart("ESP32")

In [ ]:
# TEST 3 -- Ruban LED 12V + Type-C (connecteur identique)
show_smart("Ruban LED RGB 5050")
show_smart("USB TYPE C POUR RASPBERRY")

In [ ]:
# TEST 4 -- Voiture robot | Tournevis -> rien | 74HC595 (cas du coach)
show_smart("Kit Robot Voiture")
show_smart("Coffret de tournevis")
show_smart("74HC595N", in_stock_only=False)

## Etape 7 : Regler le moteur via le CSV
Edite `regles_recommandation.csv` (attributs du meme type, complements, mots-cles) puis re-execute a partir de l'Etape 5.

In [ ]:
# Pour regler : edite regles_recommandation.csv puis re-execute a partir de l'Etape 5.
# Exemple de modif en direct :
COMPLEMENTARITY_MAP["mobilite"] = ["batterie","chargeur","moteur","carte"]
engine.comp_map = COMPLEMENTARITY_MAP
print("Carte de complementarite mise a jour")
show_smart("Drone")

## Etape 8 : Audit qualite -- tester TOUS les produits
Verifie automatiquement qu'aucune reco n'est illogique (in_stock_only=False = logique pure, comme si tout etait en stock).

In [ ]:
from collections import Counter
import time
P = engine.products
violations = []; n_test = n_sub = n_comp = n_empty = 0
t0 = time.time()
for i in range(len(P)):
    res = engine.recommend_smart(i, n=3, in_stock_only=False)
    if res is None: continue
    n_test += 1; sc = res["source_cat"]; sa = res["source_attrs"]
    subs = list(res["upgrade"].itertuples()) + list(res["alternative"].itertuples())
    for r in subs:
        if r.category != sc:
            violations.append(("substitut autre categorie", res["source_title"], r.product_title))
        for a in [x for x in TYPE_ATTRIBUTES.get(sc, []) if sa.get(x) is not None]:
            if r.attrs.get(a) != sa[a]:
                violations.append((f"substitut attribut {a}", res["source_title"], r.product_title))
    for r in res["goes_with"].itertuples():
        if r.category not in COMPLEMENTARITY_MAP.get(sc, []):
            violations.append(("complement hors-carte", res["source_title"], r.product_title))
    if subs: n_sub += 1
    if len(res["goes_with"]): n_comp += 1
    if not subs and not len(res["goes_with"]): n_empty += 1
print(f"Audit LOGIQUE PURE de {n_test} produits en {time.time()-t0:.0f}s")
print(f"   avec >=1 substitut   : {n_sub} ({100*n_sub/n_test:.0f}%)")
print(f"   avec >=1 complement  : {n_comp} ({100*n_comp/n_test:.0f}%)")
print(f"   blocs vides          : {n_empty} ({100*n_empty/n_test:.0f}%)")
print(f">>> Recommandations ILLOGIQUES : {len(violations)}")
if violations:
    for t,c in Counter(v[0] for v in violations).items(): print(f"   - {t} : {c}")
    for v in violations[:10]: print(f"   - [{v[0]}] {v[1][:36]} => {v[2][:36]}")
else:
    print("   AUCUNE -- substituts du meme type, complements coherents.")

In [ ]:
# BONUS -- familles auto-decouvertes (clustering non supervise, preuve de self-discovery)
from sklearn.cluster import KMeans
K_FAMILIES = 20
km = KMeans(n_clusters=K_FAMILIES, random_state=42, n_init=4).fit(engine.X)
engine.products["auto_family"] = km.labels_
print(f"{K_FAMILIES} familles decouvertes automatiquement par le modele (sans regles ecrites a la main).")